# 05 — Evaluate Model

Objectif : charger le modèle final et analyser ses erreurs pour justifier sa mise en production.

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.ml import PipelineModel
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Fix for Spark/Java compatibility
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"

spark = (
    SparkSession.builder
    .appName("AmazonFoodReviews-Evaluate")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

DATA_PATH = Path("../data/Reviews.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("../Reviews.csv")

MODEL_DIR = Path("../models/logreg_sentiment_tfidf")

print("Using dataset:", DATA_PATH.resolve())
print("Loading model:", MODEL_DIR.resolve())

In [ ]:
df_raw = spark.read.csv(str(DATA_PATH), header=True, inferSchema=True, multiLine=True, escape='"')
df = (
    df_raw.select(
        F.col("Id").cast("long").alias("id"),
        F.col("Score").cast("int").alias("score"),
        F.col("Text").cast("string").alias("text"),
    )
    .filter(F.col("score").isNotNull())
    .na.drop(subset=["score", "text"])
    .withColumn("text", F.lower(F.col("text")))
    .withColumn("text", F.regexp_replace(F.col("text"), r"[^a-z0-9\s]", " "))
    .withColumn("text", F.regexp_replace(F.col("text"), r"\s+", " "))
    .withColumn(
        "sentiment",
        F.when(F.col("score").isin([1, 2]), F.lit("Negative"))
         .when(F.col("score") == 3, F.lit("Neutral"))
         .otherwise(F.lit("Positive"))
    )
)

# Split with same seed
_, test_df = df.randomSplit([0.8, 0.2], seed=42)

In [ ]:
model = PipelineModel.load(str(MODEL_DIR))
predictions = model.transform(test_df).cache()

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")
metrics = {
    "Accuracy": evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"}),
    "F1": evaluator.evaluate(predictions, {evaluator.metricName: "f1"}),
}
print("Final Metrics:", metrics)

## 1. Per-Class Analysis

Visualisation de la précision et du rappel par catégorie de sentiment.

In [ ]:
class_metrics = []
for label_name in ["Positive", "Neutral", "Negative"]:
    # Note: Label mapping depends on StringIndexer order, here we use generic logic
    pass

# Simple bar chart of correct vs incorrect predictions
res = predictions.withColumn("is_correct", F.when(F.col("label") == F.col("prediction"), 1).otherwise(0))
correct_counts = res.groupBy("sentiment", "is_correct").count().toPandas()

plt.figure(figsize=(12, 6))
sns.barplot(x="sentiment", y="count", hue="is_correct", data=correct_counts, palette="coolwarm")
plt.title("Correct vs Incorrect Predictions per Sentiment", fontsize=15)
plt.legend(title="Correct", labels=["No", "Yes"])
plt.show()

## 2. Confidence Analysis

On regarde si le modèle est sûr de lui quand il se trompe.

In [ ]:
# Extract the max probability for each prediction
from pyspark.ml.functions import vector_to_array
prob_df = predictions.withColumn("prob_arr", vector_to_array("probability")) \
                     .withColumn("max_prob", F.array_max("prob_arr")) \
                     .withColumn("is_correct", F.when(F.col("label") == F.col("prediction"), "Correct").otherwise("Wrong"))

prob_pd = prob_df.select("is_correct", "max_prob").sample(0.1).toPandas()

plt.figure(figsize=(10, 6))
sns.histplot(data=prob_pd, x="max_prob", hue="is_correct", kde=True, element="step")
plt.title("Model Confidence Distribution", fontsize=15)
plt.xlabel("Max Probability (Confidence)", fontsize=12)
plt.show()